In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
path = '../../dataset_imputed.csv'
df_raw = pd.read_csv(path, header=0)

In [20]:
nulls = df_raw.isnull().sum()
null_df = pd.DataFrame({
    'Nulos'      : nulls,
    '% del total': (nulls / len(df_raw) * 100).round(2)
}).query('Nulos > 0').sort_values('% del total', ascending=False)

print(f"Columnas con nulos: {len(null_df)}")
display(null_df)

Columnas con nulos: 16


,Nulos,% del total
average.first.period,53010,68.38
failed.subject.first.period,53010,68.38
dropped.subject.first.period,53010,68.38
wellness.activities,41824,53.95
art.culture,41824,53.95
student.society.leadership,41824,53.95
athletic.sports,41824,53.95
life.work.mentoring,41824,53.95
mother.education.complete,37282,48.10
mother.education.summary,37282,48.10


El dataset tiene varios estudiantes que no estan en undergraduate level. Se quitan los que no estan.

In [21]:
df = df_raw[df_raw['level'] == 'Undergraduate'].copy()
print(f"Universitarios: {len(df):,} / {len(df_raw):,} ({len(df)/len(df_raw)*100:.1f}%)")

Universitarios: 77,517 / 77,517 (100.0%)


# K-Means
Aqui se hace la depuracion para entrenar modelos K-Means

In [22]:
# ── 1.2  Eliminar columnas con fuga o irrelevantes ─────────────────────────
DROP_COLS = [
    'student.id',
    'level',                            # constante tras filtrar
    'average.first.period',             # solo en Tec21
    'failed.subject.first.period',      # solo en Tec21
    'dropped.subject.first.period',     # solo en Tec21
    'dropout.semester',                 # derivado de retention → fuga
    'scholarship.type',                 # redundante con total.scholarship.loan
    'scholarship.perc',                 # redundante porque se tiene total.scholarship.loan
    'program',
    'gender',                           # se quita ya que no aporta mucho
    'id.school.origin',                 # se elimina 
    'school.cost',                      # redundante con socioeconomic.level
    'general.math.eval',                # escala inconsistente, baja cobertura. 50% no lo tienen
    'father.exatec', 'mother.exatec',   # redundantes
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    'scholarship.perc', 'loan.perc',    # ya están en total.scholarship.loan
    'zone.type',                        # muchos missing
    'socioeconomic.level',              # muchos missing
    'social.lag',                       # muchos missing
]
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

'parents.exatec' se cambia a 0 o 1: donde 1 tiene papas exatec. 0 si tiene 'No' o 'No information'.

In [23]:
df['parents_exatec_enc'] = df['parents.exatec'].map({
    'Yes': 1,
    'No': 0,
    'No information': 0
})
df.drop(columns=['parents.exatec'], inplace=True)

##### MAR Missing At Random: los datos faltantes tienen un patrón, pero ese patrón depende de otras variables que sí puedes observar.
AD14-AD17: tenían 3 columnas (physical.education, cultural.diffusion, student.society)
AD18-AD20: cambiaron a 5 columnas LiFE (athletic.sports, art.culture, student.society.leadership, life.work.mentoring, wellness.activities) con total.life.activities como contador.

Se crean tres variables binarias que indican si el alumno participó en alguna actividad de cada categoría: 
- física (has_physical) si tiene physical.education, athletic.sports, o wellness.activities. 
- cultural (has_cultural) si tiene cultural.diffusion, o art.culture.
- social (has_society) si tiene student.society, student.society.leadership, o life.work.mentoring. 

Esto reemplaza las 9 columnas originales de actividades, que eran inconsistentes entre generaciones por el cambio de modelo educativo de PreTec21 a Tec21.

In [24]:
# ── 1.4  MAR: bandera binaria has_extracurriculars ──────────────────────────
ACTIVITY_COLS = [
    'physical.education', 'cultural.diffusion', 'student.society',
    'total.life.activities', 'athletic.sports', 'art.culture',
    'student.society.leadership', 'life.work.mentoring', 'wellness.activities'
]

def parse_activity(val):
    """Devuelve 1 si el estudiante participó en alguna actividad, 0 en otro caso."""
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ('Does not apply', 'No information', '', '0', '0.0'):
        return 0
    try:
        return int(float(s) > 0)
    except Exception:
        return 0

df['has_extracurriculars'] = df['total.life.activities'].apply(parse_activity)
df.drop(columns=[c for c in ACTIVITY_COLS if c in df.columns], inplace=True)
print(f"has_extracurriculars  →  con actividades: {df['has_extracurriculars'].sum():,} ({df['has_extracurriculars'].mean()*100:.1f}%)")

has_extracurriculars  →  con actividades: 72,195 (93.1%)


##### MNAR - Missing Not At Random: los datos faltantes no son aleatorios, sino que la ausencia del dato en sí misma tiene significado.
Aqui, la columna 'first.generation' tiene tres opciones:
1. Yes: First.generation.yes
2. No: First.generation.no
3. No information/Does not apply: First.generation.No.Information

In [25]:
# first.generation → 3 columnas binarias (one-hot)
df['first.generation.yes']            = (df['first.generation'].str.strip() == 'Yes').astype(int)
df['first.generation.no']             = (df['first.generation'].str.strip() == 'No').astype(int)
df['first.generation.no.information'] = (~df['first.generation'].str.strip().isin(['Yes', 'No'])).astype(int)
df.drop(columns=['first.generation'], inplace=True)

print("Distribución:")
print(f"  first.generation.yes:            {df['first.generation.yes'].sum():>6,} ({df['first.generation.yes'].mean()*100:.1f}%)")
print(f"  first.generation.no:             {df['first.generation.no'].sum():>6,} ({df['first.generation.no'].mean()*100:.1f}%)")
print(f"  first.generation.no.information: {df['first.generation.no.information'].sum():>6,} ({df['first.generation.no.information'].mean()*100:.1f}%)")

Distribución:
  first.generation.yes:             5,840 (7.5%)
  first.generation.no:             71,677 (92.5%)
  first.generation.no.information:      0 (0.0%)


Se cambia tec.no.tec a si ha estado en prepa tec o no.

In [26]:
df['estuvo.prepa_tec'] = df['tec.no.tec'].map({'TEC': 1, 'NO TEC': 0}).astype(int)
df.drop(columns=['tec.no.tec'], inplace=True)

Se reemplaza la columna foreign por dos columnas binarias (foreign_Yes: National y foreign_Yes: Foreigner), donde cada una vale 1 si el alumno pertenece a esa categoría y 0 si no es foreigner.

In [27]:
foreign_dummies = pd.get_dummies(df['foreign'], prefix='foreign', drop_first=True, dtype=int)
df = pd.concat([df, foreign_dummies], axis=1)
df.drop(columns=['foreign'], inplace=True)

Crea un one hot encoding para cada escuela del Tec. Se crea cada columna por escuela (i.e. Ingenieria, Negocios, etc.).

In [28]:
# school
school_dummies = pd.get_dummies(df['school'], prefix='school', drop_first=False, dtype=int)
df = pd.concat([df, school_dummies], axis=1)
df.drop(columns=['school'], inplace=True)

#### Creacion de archivo CSV
Se guardan estos datos en un csv

In [29]:
df.to_csv('../dataset_k_means.csv', index=False)
print("Guardado en ../dataset_k_means.csv")

Guardado en ../dataset_k_means.csv
